In [1]:
import sys
import os
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(project_root)
from datetime import datetime

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import HashingVectorizer, TfidfVectorizer
import joblib
from tqdm import tqdm
from concurrent.futures import as_completed
from common.const import (
    SCHEMA,
    TABLE_RAW_REVIEWS,
    TABLE_FEATURED_USER_VECTORS,
    TABLE_USER_AGGREGATED_STATE,
    TABLE_MATERIALIZED_VIEW_USER,
    TABLE_USER_ITEM_STATE,          
    TABLE_USER_ITEM_STATE_SNAPSHOT,
    TABLE_FEATURED_DENSE_VECTORS,   
)
from common.helper import connect_clickhouse, clickhouse_extract_data_as_df
from concurrent.futures import ThreadPoolExecutor
import threading

c:\Users\Admin\.conda\envs\scc454\lib\site-packages\torchvision\io\image.py:13: UserWarning: Failed to load image Python extension: '[WinError 127] The specified procedure could not be found'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(


In [11]:
def drop_clickhouse_tables():
    client = connect_clickhouse()
    # client.command(f"DROP TABLE IF EXISTS {SCHEMA}.{TABLE_USER_AGGREGATED_STATE}")
    # client.command(f"DROP TABLE IF EXISTS {SCHEMA}.{TABLE_USER_ITEM_STATE}")
    client.command(f"DROP TABLE IF EXISTS {SCHEMA}.user_item_state_merged")
    # client.command(f"DROP TABLE IF EXISTS {SCHEMA}.{TABLE_MATERIALIZED_VIEW_USER}")
    client.close()
drop_clickhouse_tables()

In [ ]:
from common.const import SCHEMA, TABLE_RAW_PRODUCTS, DEVICE
from common.helper import clickhouse_extract_data_as_df

# Products
products_query = f"""
SELECT *
FROM {SCHEMA}.{TABLE_RAW_PRODUCTS}
WHERE rand() % 100 = 0
LIMIT 10000
"""
products_df = clickhouse_extract_data_as_df(products_query)

# Reviews
reviews_query = f"""
SELECT *
FROM {SCHEMA}.{TABLE_RAW_REVIEWS}
WHERE rand() % 100 = 0
LIMIT 10000
"""
reviews_df = clickhouse_extract_data_as_df(reviews_query)

connected to clickhouse
connected to clickhouse


In [7]:
products_df

,main_category,title,average_rating,rating_number,features,description,price,first_image,image_count,variant_count,videos,store,categories,details,parent_asin,bought_together,subtitle,author,product_category
0,Toys & Games,"YOTTOY Magalina Dog 8"" Soft Toy",3.0,3,Colorfully patterned in soft chenille Magalina...,"Product Description Magalina Dog 8"" Soft Toy F...",35.99,https://m.media-amazon.com/images/I/81xuRghO2c...,1,1,"{""title"": [], ""url"": [], ""user_id"": []}",YOTTOY,Toys & Games Preschool Pre-Kindergarten Toys S...,"{""Product Dimensions"": ""6.5 x 4.5 x 8 inches"",...",0439843073,<NA>,<NA>,<NA>,Toys_and_Games
1,Toys & Games,Clemens Habicht 540 Colors 3D Sphere Puzzle,4.4,27,<NA>,Features 540 piecesDurable plasticDisplay stan...,None,https://m.media-amazon.com/images/I/81mo+c16LZ...,9,9,"{""title"": [], ""url"": [], ""user_id"": []}",Clemens Habicht Colour Puzzles,<NA>,"{""Product Dimensions"": ""7.6 x 7.75 x 1.9 inche...",0648342646,<NA>,<NA>,<NA>,Toys_and_Games
2,Toys & Games,Mudpuppy Zoo Animals Book of Stickers,5.0,1,8 sheets of stickers 70+ stickers per sticker ...,Product Description Emilie Chollat has illustr...,6.00,https://m.media-amazon.com/images/I/71OaVs7jjs...,1,1,"{""title"": [], ""url"": [], ""user_id"": []}",Mudpuppy,Toys & Games Preschool Pre-Kindergarten Toys A...,"{""Product Dimensions"": ""6.5 x 0.5 x 4.5 inches...",0735327408,<NA>,<NA>,<NA>,Toys_and_Games
3,Toys & Games,Mudpuppy Unicorn Magic Puzzle Sticks,4.5,59,<NA>,<NA>,None,<NA>,0,0,"{""title"": [""PLAYQID Enchanted World of Unicorn...",by Mudpuppy,Toys & Games Preschool Pre-Kindergarten Toys,<NA>,073536043X,<NA>,<NA>,<NA>,Toys_and_Games
4,Toys & Games,Butterflies Illuminated 500 Piece Glow in the ...,4.7,106,500-PIECE PUZZLE – The 500-piece Butterflies I...,Piece together a colorful nature scene with Mu...,14.99,https://m.media-amazon.com/images/I/81cD9gQpg1...,9,9,"{""title"": [""MasterPieces, Inc. Video""], ""url"":...",Mudpuppy,Toys & Games Puzzles Jigsaw Puzzles,"{""Product Dimensions"": ""20 x 20 x 0.1 inches"",...",0735366640,<NA>,<NA>,<NA>,Toys_and_Games
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,Toys & Games,MagiDeal Balloon Clip Ties for Helium Balloons...,3.3,20,and high quality Useful tool for balloon decor...,Description: - and high quality - Useful tool ...,None,https://m.media-amazon.com/images/I/51R-Ge7HCi...,9,9,"{""title"": [], ""url"": [], ""user_id"": []}",MagiDeal,Toys & Games Party Supplies Decorations Balloons,"{""Item model number"": ""STK0155004320"", ""Best S...",B01H1I2CLS,<NA>,<NA>,<NA>,Toys_and_Games
9996,Toys & Games,DiDi/iRC 12mm Hex Adapter Aluminum Hex Wheel H...,4.0,60,12mm hex wheel hubs w/pins Fit for 1/18 Wltoys...,"Features: 1,Upgrade your 1:18th RC car' Wheel ...",6.98,https://m.media-amazon.com/images/I/51tED7PA8n...,5,5,"{""title"": [], ""url"": [], ""user_id"": []}",DiDi/iRC,<NA>,"{""Package Dimensions"": ""3.03 x 2.48 x 0.08 inc...",B01H26YNLG,<NA>,<NA>,<NA>,Toys_and_Games
9997,Cell Phones & Accessories,"Galaxy On5 Case, Evocel [New Generation Series...",4.4,764,"[Dependable Protection] Heavy duty, dual layer...",<NA>,None,https://m.media-amazon.com/images/I/71XnszTk0D...,8,8,"{""title"": [], ""url"": [], ""user_id"": []}",Evocel,"Cell Phones & Accessories Cases, Holsters & Sl...","{""Package Dimensions"": ""8 x 7.4 x 1.2 inches"",...",B01H2AJB9G,<NA>,<NA>,<NA>,Cell_Phones_and_Accessories
9998,Toys & Games,50 Direct-Fit Airtight 40.6mm Capsule Holders ...,5.0,4,<NA>,<NA>,19.95,https://m.media-amazon.com/images/I/81uIXGKtEj...,2,2,"{""title"": [], ""url"": [], ""user_id"": []}",USA,Toys & Games Games & Accessories Casino Equipm...,"{""Manufacturer"": ""USA""}",B01H2AWGX4,<NA>,<NA>,<NA>,Toys_and_Games


In [9]:
import numpy as np
import pandas as pd

def basic_eda(products_query: str, reviews_query: str):
    """
    Run basic EDA on randomly sampled products & reviews subset.
    Returns a dictionary summary.
    """

    # ── Load data from ClickHouse ─────────────────────────────
    products_df = clickhouse_extract_data_as_df(products_query)
    reviews_df = clickhouse_extract_data_as_df(reviews_query)

    summary = {}

    # ── PRODUCTS EDA ──────────────────────────────────────────
    summary["num_products"] = len(products_df)
    summary["product_missing_ratio"] = products_df.isna().mean().to_dict()

    if "price" in products_df.columns:
        summary["price_stats"] = products_df["price"].describe().to_dict()

    if "average_rating" in products_df.columns:
        summary["product_rating_stats"] = products_df["average_rating"].describe().to_dict()

    if "main_category" in products_df.columns:
        summary["category_distribution"] = (
            products_df["main_category"]
            .value_counts(normalize=True)
            .to_dict()
        )

    # Text length stats
    if "title" in products_df.columns:
        products_df["title_len"] = products_df["title"].fillna("").str.len()
        summary["title_length_stats"] = products_df["title_len"].describe().to_dict()

    if "description" in products_df.columns:
        products_df["desc_len"] = products_df["description"].fillna("").str.len()
        summary["description_length_stats"] = products_df["desc_len"].describe().to_dict()

    # ── REVIEWS EDA ───────────────────────────────────────────
    summary["num_reviews"] = len(reviews_df)
    summary["review_missing_ratio"] = reviews_df.isna().mean().to_dict()

    if "rating" in reviews_df.columns:
        summary["review_rating_distribution"] = (
            reviews_df["rating"].value_counts(normalize=True)
            .sort_index()
            .to_dict()
        )

    if "verified_purchase" in reviews_df.columns:
        summary["verified_purchase_ratio"] = (
            reviews_df["verified_purchase"]
            .value_counts(normalize=True)
            .to_dict()
        )

    # User & product interaction stats
    if "user_id" in reviews_df.columns:
        summary["unique_users"] = reviews_df["user_id"].nunique()

    if "parent_asin" in reviews_df.columns:
        summary["unique_products_in_reviews"] = reviews_df["parent_asin"].nunique()

        if summary["unique_products_in_reviews"] > 0:
            summary["avg_reviews_per_product"] = (
                len(reviews_df) / summary["unique_products_in_reviews"]
            )

    if "user_id" in reviews_df.columns and summary["unique_users"] > 0:
        summary["avg_reviews_per_user"] = (
            len(reviews_df) / summary["unique_users"]
        )

    # Review text length
    if "text" in reviews_df.columns:
        reviews_df["text_len"] = reviews_df["text"].fillna("").str.len()
        summary["review_text_length_stats"] = (
            reviews_df["text_len"].describe().to_dict()
        )

    return summary

In [10]:
NUM_RUNS = 5
all_summaries = []

for i in range(NUM_RUNS):
    print(f"Running EDA sample {i+1}/{NUM_RUNS}")

    products_query = f"""
    SELECT *
    FROM {SCHEMA}.{TABLE_RAW_PRODUCTS}
    ORDER BY rand()
    LIMIT 10000
    """

    reviews_query = f"""
    SELECT *
    FROM {SCHEMA}.{TABLE_RAW_REVIEWS}
    ORDER BY rand()
    LIMIT 50000
    """

    summary = basic_eda(products_query, reviews_query)
    all_summaries.append(summary)

Running EDA sample 1/5
connected to clickhouse
connected to clickhouse
Running EDA sample 2/5
connected to clickhouse
connected to clickhouse
Running EDA sample 3/5
connected to clickhouse
connected to clickhouse
Running EDA sample 4/5
connected to clickhouse
connected to clickhouse
Running EDA sample 5/5
connected to clickhouse
connected to clickhouse


In [11]:
def aggregate_numeric_stat(all_summaries, key, stat):
    values = [
        s[key][stat]
        for s in all_summaries
        if key in s and stat in s[key]
    ]
    return np.mean(values)

avg_price_mean = aggregate_numeric_stat(all_summaries, "price_stats", "mean")
print("Average sampled price mean:", avg_price_mean)

Average sampled price mean: nan


c:\Users\Admin\.conda\envs\scc454\lib\site-packages\numpy\core\fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\Admin\.conda\envs\scc454\lib\site-packages\numpy\core\_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


In [12]:
all_summaries

[{'num_products': 10000,
  'product_missing_ratio': {'main_category': 0.0663,
   'title': 0.0,
   'average_rating': 0.0,
   'rating_number': 0.0,
   'features': 0.2856,
   'description': 0.4142,
   'price': 0.6422,
   'first_image': 0.155,
   'image_count': 0.0,
   'variant_count': 0.0,
   'videos': 0.0,
   'store': 0.0138,
   'categories': 0.1258,
   'details': 0.0185,
   'parent_asin': 0.0,
   'bought_together': 1.0,
   'subtitle': 0.9994,
   'author': 0.9998,
   'product_category': 0.0},
  'price_stats': {'count': 3578,
   'unique': 1366,
   'top': Decimal('9.99'),
   'freq': 129},
  'product_rating_stats': {'count': 10000.0,
   'mean': 4.12585973739624,
   'std': 0.7978091835975647,
   'min': 1.0,
   '25%': 3.799999952316284,
   '50%': 4.300000190734863,
   '75%': 4.699999809265137,
   'max': 5.0},
  'category_distribution': {'Cell Phones & Accessories': 0.2298382778194281,
   'Toys & Games': 0.14629966798757632,
   'Computers': 0.07861197386740923,
   'All Electronics': 0.07786226